# EXP22: 点差敏感度全景 + 盈亏平衡点

精细扫描点差 $0.10-$1.00 步长 $0.05，找到策略盈亏平衡点差。
同时测试不同点差下最优参数是否漂移。

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
}
print('Ready')

In [ ]:
spreads = np.arange(0.10, 1.05, 0.05)
results = []
for sp in spreads:
    sp = round(sp, 2)
    r = run_variant(data, f"sp=${sp:.2f}", **{**LIVE_KWARGS, "spread_cost": sp})
    r['spread'] = sp
    results.append(r)
    print(f"  sp=${sp:.2f}: Sharpe={r['sharpe']:.2f}, PnL=${r['total_pnl']:.0f}, N={r['n']}")

df = pd.DataFrame([{'spread': r['spread'], 'sharpe': r['sharpe'],
    'total_pnl': r['total_pnl'], 'n': r['n'], 'max_dd': r.get('max_dd', 0)} for r in results])
print(df.to_string(index=False))

In [ ]:
# 找盈亏平衡点
breakeven_idx = None
for i in range(len(df)-1):
    if df.iloc[i]['total_pnl'] > 0 and df.iloc[i+1]['total_pnl'] <= 0:
        # 线性插值
        p1, p2 = df.iloc[i]['total_pnl'], df.iloc[i+1]['total_pnl']
        s1, s2 = df.iloc[i]['spread'], df.iloc[i+1]['spread']
        breakeven = s1 + (s2-s1) * p1 / (p1-p2)
        print(f"\nBreakeven spread: ~${breakeven:.3f}")
        breakeven_idx = i
        break

if breakeven_idx is None:
    if df.iloc[-1]['total_pnl'] > 0:
        print("Strategy profitable at ALL spread levels tested!")
    else:
        print(f"Strategy already unprofitable at sp=${df.iloc[0]['spread']:.2f}")

# Sharpe = 0 交叉点
for i in range(len(df)-1):
    if df.iloc[i]['sharpe'] > 0 and df.iloc[i+1]['sharpe'] <= 0:
        s1, s2 = df.iloc[i]['spread'], df.iloc[i+1]['spread']
        sh1, sh2 = df.iloc[i]['sharpe'], df.iloc[i+1]['sharpe']
        be = s1 + (s2-s1) * sh1 / (sh1-sh2)
        print(f"Sharpe=0 breakeven: ~${be:.3f}")

In [ ]:
# 每 $0.10 算边际成本
print("\n=== Marginal Cost Per $0.10 Spread ===")
for i in range(1, len(df)):
    if df.iloc[i]['spread'] in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]:
        delta = df.iloc[i]['total_pnl'] - df.iloc[i-1]['total_pnl']
        print(f"  ${df.iloc[i-1]['spread']:.2f} -> ${df.iloc[i]['spread']:.2f}: PnL delta = ${delta:.0f}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp22_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved')